# Gold Analytical Model

## Purpose

This notebook transforms analysis-ready Silver data into a
business-focused Gold dimensional model.

The Gold layer is designed to support:

- Business reporting
- Direct Lake semantic modeling
- DAX measures
- Power BI dashboards
- Regional and programme analysis
- Educator activity analysis

## Gold Tables

- `gold_fact_event_delivery`
- `gold_dim_date`
- `gold_dim_location`
- `gold_dim_educator`
- `gold_bridge_event_educator`

## Grain

`gold_fact_event_delivery` contains one row per event delivery record.

`event_group_id` identifies related delivery records belonging to the
same overall event or programme activity.

In [2]:
# Import

from pyspark.sql.functions import (
    col,
    lit,
    when,
    coalesce,
    current_timestamp,
    date_format,
    year,
    month,
    quarter,
    weekofyear,
    dayofmonth,
    explode,
    sequence,
    min as spark_min,
    max as spark_max,
    countDistinct,
    concat,
    round as spark_round
)

StatementMeta(, 308a472c-add8-4444-bd20-b0aca360c779, 4, Finished, Available, Finished, False)

In [3]:
# Read Silver tables

silver_events_df = spark.table("silver_events")
silver_locations_df = spark.table("silver_locations")
silver_educators_df = spark.table("silver_educators")
silver_event_educators_df = spark.table(
    "silver_event_educators"
)

print("Silver tables loaded successfully.")
print("----------------------------------------")
print(f"Events: {silver_events_df.count()}")
print(f"Locations: {silver_locations_df.count()}")
print(f"Educators: {silver_educators_df.count()}")
print(
    f"Event educators: "
    f"{silver_event_educators_df.count()}"
)

StatementMeta(, 308a472c-add8-4444-bd20-b0aca360c779, 5, Finished, Available, Finished, False)

Silver tables loaded successfully.
----------------------------------------
Events: 47
Locations: 27
Educators: 8
Event educators: 132


In [4]:
# Create core Fact Table

gold_fact_event_delivery_df = (
    silver_events_df

    # Date key for dimensional model
    .withColumn(
        "event_date_key",
        date_format(
            col("event_date"),
            "yyyyMMdd"
        ).cast("int")
    )

    # Unified people-reached metric
    .withColumn(
        "people_reached",
        coalesce(
            col("engaged_people_count"),
            col("attended_count"),
            lit(0)
        )
    )

    # One record represents one delivery
    .withColumn(
        "delivery_count",
        lit(1)
    )

    # Attendance difference
    .withColumn(
        "attendance_variance",
        when(
            col("registered_count").isNotNull()
            & col("attended_count").isNotNull(),
            col("attended_count")
            - col("registered_count")
        )
    )

    # Attendance rate can legitimately exceed 100%
    .withColumn(
        "attendance_rate",
        when(
            col("registered_count").isNotNull()
            & (col("registered_count") > 0)
            & col("attended_count").isNotNull(),

            spark_round(
                col("attended_count")
                / col("registered_count"),
                4
            )
        )
    )

    # Convert duration to hours
    .withColumn(
        "session_duration_hours",
        spark_round(
            col("session_duration_minutes") / 60.0,
            2
        )
    )

    .withColumn(
        "gold_processed_timestamp",
        current_timestamp()
    )

    .select(
        "event_id",
        "event_group_id",
        "event_date_key",
        "event_date",
        "location_id",
        "lead_educator_id",

        "event_name",
        "event_type",
        "programme_type",

        "delivery_language",
        "audience_type",
        "delivery_format",
        "demonstration_type",

        "registration_required",
        "registered_count",
        "attended_count",
        "attendance_variance",
        "attendance_rate",

        "engaged_people_count",
        "people_reached",
        "engaged_boats_count",

        "session_duration_minutes",
        "session_duration_hours",

        "satisfaction_score",
        "engagement_level",

        "weather_condition",
        "is_estimated",
        "event_status",

        "delivery_count",
        "gold_processed_timestamp"
    )
)

display(gold_fact_event_delivery_df.limit(10))

StatementMeta(, 308a472c-add8-4444-bd20-b0aca360c779, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 3e18495b-a0f1-4d46-ad80-1a518c553761)

In [5]:
# Location Dimension

gold_dim_location_df = (
    silver_locations_df

    # create business-friendly venue category
    .withColumn(
        "venue_category",
        when(
            col("venue_type").isin(
                "Church",
                "Church Hall"
            ),
            lit("Church")
        ).otherwise(
            col("venue_type")
        )
    )

    .select(
        "location_id",
        "venue_name",
        "suburb",
        "region",
        "venue_type",
        "venue_category",
        "indoor_outdoor",
        "water_access"
    )
    .dropDuplicates(["location_id"])
    .withColumn(
        "gold_processed_timestamp",
        current_timestamp()
    )
)

display(gold_dim_location_df)

StatementMeta(, 308a472c-add8-4444-bd20-b0aca360c779, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 38a2f13f-d359-400a-91e0-900a463985ae)

In [6]:
# Educator Dimension

gold_dim_educator_df = (
    silver_educators_df
    .select(
        "educator_id",
        "educator_name",
        "primary_language",
        "secondary_language",
        "experience_level",
        "active_status"
    )
    .dropDuplicates(["educator_id"])
    .withColumn(
        "gold_processed_timestamp",
        current_timestamp()
    )
)

display(gold_dim_educator_df)

StatementMeta(, 308a472c-add8-4444-bd20-b0aca360c779, 8, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 679e4d4e-d24f-43a0-a3ec-8fd6add53e83)

In [7]:
# Date Dimension

date_bounds = (
    silver_events_df
    .agg(
        spark_min("event_date").alias("min_date"),
        spark_max("event_date").alias("max_date")
    )
    .first()
)

date_range_df = spark.createDataFrame(
    [
        (
            date_bounds["min_date"],
            date_bounds["max_date"]
        )
    ],
    [
        "start_date",
        "end_date"
    ]
)

gold_dim_date_df = (
    date_range_df

    .select(
        explode(
            sequence(
                col("start_date"),
                col("end_date")
            )
        ).alias("date")
    )

    .withColumn(
        "date_key",
        date_format(
            col("date"),
            "yyyyMMdd"
        ).cast("int")
    )

    .withColumn(
        "year",
        year("date")
    )

    .withColumn(
        "quarter",
        concat(
            lit("Q"),
            quarter("date")
        )
    )

    .withColumn(
        "month_number",
        month("date")
    )

    .withColumn(
        "month_name",
        date_format(
            col("date"),
            "MMMM"
        )
    )

    .withColumn(
        "year_month",
        date_format(
            col("date"),
            "yyyy-MM"
        )
    )

    .withColumn(
        "week_of_year",
        weekofyear("date")
    )

    .withColumn(
        "day_of_month",
        dayofmonth("date")
    )

    .withColumn(
        "day_name",
        date_format(
            col("date"),
            "EEEE"
        )
    )

    .select(
        "date_key",
        "date",
        "year",
        "quarter",
        "month_number",
        "month_name",
        "year_month",
        "week_of_year",
        "day_of_month",
        "day_name"
    )
)

display(gold_dim_date_df.limit(10))

StatementMeta(, 308a472c-add8-4444-bd20-b0aca360c779, 9, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 3daaaeec-ae7f-448c-9e88-b1b404409684)

In [8]:
# Event - Educator Bridge

gold_bridge_event_educator_df = (
    silver_event_educators_df

    .select(
        "event_id",
        "educator_id",
        "educator_role",
        "hours_worked",
        "is_estimated" 
    )

    .dropDuplicates([
        "event_id",
        "educator_id"
    ])

    .withColumn(
        "gold_processed_timestamp",
        current_timestamp()
    )
)

display(
    gold_bridge_event_educator_df.limit(20)
)

StatementMeta(, 308a472c-add8-4444-bd20-b0aca360c779, 10, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 035a164e-662f-46dd-84b7-45b52954ec6c)

In [9]:
# Gold Validation

fact_count = gold_fact_event_delivery_df.count()
location_count = gold_dim_location_df.count()
educator_count = gold_dim_educator_df.count()
bridge_count = gold_bridge_event_educator_df.count()
date_count = gold_dim_date_df.count()

event_group_count = (
    gold_fact_event_delivery_df
    .select("event_group_id")
    .distinct()
    .count()
)

events_without_locations = (
    gold_fact_event_delivery_df
    .select("location_id")
    .join(
        gold_dim_location_df.select("location_id"),
        on="location_id",
        how="left_anti"
    )
    .count()
)

bridge_without_events = (
    gold_bridge_event_educator_df
    .select("event_id")
    .join(
        gold_fact_event_delivery_df.select("event_id"),
        on="event_id",
        how="left_anti"
    )
    .count()
)

bridge_without_educators = (
    gold_bridge_event_educator_df
    .select("educator_id")
    .join(
        gold_dim_educator_df.select("educator_id"),
        on="educator_id",
        how="left_anti"
    )
    .count()
)

events_without_staff = (
    gold_fact_event_delivery_df
    .select("event_id")
    .join(
        gold_bridge_event_educator_df
        .select("event_id")
        .distinct(),
        on="event_id",
        how="left_anti"
    )
    .count()
)

print("Gold Data Model Validation")
print("----------------------------------------")
print(f"Delivery records: {fact_count}")
print(f"Event groups: {event_group_count}")
print(f"Locations: {location_count}")
print(f"Educators: {educator_count}")
print(f"Event-educator records: {bridge_count}")
print(f"Date records: {date_count}")
print("----------------------------------------")
print(
    f"Events without valid location: "
    f"{events_without_locations}"
)
print(
    f"Bridge records without event: "
    f"{bridge_without_events}"
)
print(
    f"Bridge records without educator: "
    f"{bridge_without_educators}"
)
print(
    f"Events without educator: "
    f"{events_without_staff}"
)

assert fact_count == 47
assert event_group_count == 34
assert location_count == 27
assert educator_count == 8
assert bridge_count == 132

assert events_without_locations == 0
assert bridge_without_events == 0
assert bridge_without_educators == 0
assert events_without_staff == 0

print("----------------------------------------")
print("All Gold validation checks passed.")

StatementMeta(, 308a472c-add8-4444-bd20-b0aca360c779, 11, Finished, Available, Finished, False)

Gold Data Model Validation
----------------------------------------
Delivery records: 47
Event groups: 34
Locations: 27
Educators: 8
Event-educator records: 132
Date records: 166
----------------------------------------
Events without valid location: 0
Bridge records without event: 0
Bridge records without educator: 0
Events without educator: 0
----------------------------------------
All Gold validation checks passed.


In [10]:
# Write into Gold Delta Tables

def write_gold_table(dataframe, table_name):
    (
        dataframe.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(table_name)
    )

    print(f"Created Gold table: {table_name}")

write_gold_table(
    gold_fact_event_delivery_df,
    "gold_fact_event_delivery"
)

write_gold_table(
    gold_dim_date_df,
    "gold_dim_date"
)

write_gold_table(
    gold_dim_location_df,
    "gold_dim_location"
)

write_gold_table(
    gold_dim_educator_df,
    "gold_dim_educator"
)

write_gold_table(
    gold_bridge_event_educator_df,
    "gold_bridge_event_educator"
)

StatementMeta(, 308a472c-add8-4444-bd20-b0aca360c779, 12, Finished, Available, Finished, False)

Created Gold table: gold_fact_event_delivery
Created Gold table: gold_dim_date
Created Gold table: gold_dim_location
Created Gold table: gold_dim_educator
Created Gold table: gold_bridge_event_educator
